# Python 3.15 Data Engineering — Phase 1 & 2: Core Stack Validation

**Author:** Dr. Ceasar Jackson Jr.  
**Platform:** macOS 26.5 ARM64  
**Python:** 3.15.0b1  
**Environment:** uv · `.venv`  

---

This notebook validates the Phase 1 runtime environment and Phase 2 data engineering lite stack interactively.  
It mirrors `scripts/validate_core.py` and `scripts/validate_stack.py` with inline output and visual confirmation.

**Sections**
1. Runtime & environment checks
2. NumPy
3. Pandas
4. Polars
5. DuckDB
6. SQLAlchemy
7. Pydantic
8. Matplotlib
9. Plotly
10. Summary

## 1. Runtime & Environment

In [ ]:
import sys, os, platform

print(f"Python   : {sys.version}")
print(f"Platform : {platform.platform()}")
print(f"Machine  : {platform.machine()}")
print(f"Prefix   : {sys.prefix}")
print(f"CWD      : {os.getcwd()}")

assert sys.version_info >= (3, 15), "Requires Python >= 3.15"
print("\n✅ Python 3.15+ confirmed")

In [ ]:
# Verify virtual environment is active
venv = os.environ.get("VIRTUAL_ENV", "")
assert venv, "No virtual environment active — run: source .venv/bin/activate"
print(f"✅ Virtual environment: {venv}")

## 2. NumPy

In [ ]:
import numpy as np
print(f"numpy {np.__version__}")

# Vectorised arithmetic
a = np.arange(1_000_000, dtype=np.float64)
result = np.sqrt(a).mean()
print(f"sqrt(arange(1M)).mean() = {result:.4f}")

# Array creation and dtype inference
mat = np.random.default_rng(42).standard_normal((500, 500))
print(f"Random matrix shape: {mat.shape}  dtype: {mat.dtype}")
print(f"Eigenvalue sum (abs): {np.abs(np.linalg.eigvals(mat[:50,:50])).sum():.2f}")
print("✅ NumPy OK")

## 3. Pandas

In [ ]:
import pandas as pd
print(f"pandas {pd.__version__}")

df = pd.DataFrame({
    "region":   ["north", "south", "north", "south", "north", "east", "west", "east"],
    "product":  ["A", "B", "A", "A", "B", "B", "A", "B"],
    "sales":    [120, 85, 200, 95, 160, 310, 75, 220],
    "returns":  [5, 3, 8, 2, 7, 12, 1, 9],
    "quarter":  ["Q1","Q1","Q2","Q2","Q3","Q3","Q4","Q4"],
})

print("\nDataFrame:")
display(df)

summary = df.groupby(["region", "product"]).agg(
    total_sales=("sales", "sum"),
    total_returns=("returns", "sum"),
    net=("sales", lambda x: x.sum() - df.loc[x.index, "returns"].sum()),
).reset_index()

print("\nGroupby summary:")
display(summary)
print("✅ Pandas OK")

## 4. Polars

In [ ]:
import polars as pl
print(f"polars {pl.__version__}")

lf = pl.LazyFrame({
    "product":  ["A", "B", "C", "A", "B", "C", "A", "B"],
    "region":   ["north", "south", "east", "north", "west", "south", "east", "north"],
    "qty":      [10, 20, 5, 15, 25, 8, 12, 30],
    "price":    [1.5, 2.0, 3.5, 1.5, 2.0, 3.5, 1.5, 2.0],
})

result = (
    lf
    .filter(pl.col("qty") > 10)
    .with_columns((pl.col("qty") * pl.col("price")).alias("revenue"))
    .group_by("product")
    .agg(
        pl.col("revenue").sum().alias("total_revenue"),
        pl.col("qty").mean().alias("avg_qty"),
    )
    .sort("total_revenue", descending=True)
    .collect()
)

print("\nPolars LazyFrame result:")
print(result)
print("✅ Polars OK")

## 5. DuckDB

In [ ]:
import duckdb
print(f"duckdb {duckdb.__version__}")

con = duckdb.connect(":memory:")

con.execute("""
    CREATE TABLE sales (
        id       INTEGER,
        region   VARCHAR,
        product  VARCHAR,
        amount   DOUBLE,
        quarter  VARCHAR
    )
""")

con.executemany("INSERT INTO sales VALUES (?, ?, ?, ?, ?)", [
    (1, "west",  "A", 1500.0, "Q1"),
    (2, "east",  "B", 2200.0, "Q1"),
    (3, "west",  "A",  800.0, "Q2"),
    (4, "north", "C", 3100.0, "Q2"),
    (5, "south", "B", 1750.0, "Q3"),
    (6, "north", "A", 4200.0, "Q3"),
    (7, "east",  "C", 2900.0, "Q4"),
    (8, "south", "A", 1100.0, "Q4"),
])

result = con.execute("""
    SELECT
        region,
        SUM(amount)   AS total,
        AVG(amount)   AS avg,
        COUNT(*)      AS txns
    FROM sales
    GROUP BY region
    ORDER BY total DESC
""").df()  # Return as Pandas DataFrame

print("\nDuckDB aggregation result:")
display(result)

# Window function
window_result = con.execute("""
    SELECT
        quarter,
        region,
        amount,
        SUM(amount) OVER (PARTITION BY quarter) AS quarter_total,
        ROUND(amount / SUM(amount) OVER (PARTITION BY quarter) * 100, 1) AS pct_of_quarter
    FROM sales
    ORDER BY quarter, amount DESC
""").df()

print("\nWindow function result:")
display(window_result)
con.close()
print("✅ DuckDB OK")

## 6. SQLAlchemy

In [ ]:
import sqlalchemy as sa
print(f"sqlalchemy {sa.__version__}")

engine = sa.create_engine("sqlite:///:memory:", echo=False)
metadata = sa.MetaData()

pipeline_runs = sa.Table(
    "pipeline_runs", metadata,
    sa.Column("id",          sa.Integer,  primary_key=True, autoincrement=True),
    sa.Column("pipeline",    sa.String,   nullable=False),
    sa.Column("status",      sa.String,   nullable=False),
    sa.Column("rows_in",     sa.Integer),
    sa.Column("rows_out",    sa.Integer),
    sa.Column("duration_s",  sa.Float),
)
metadata.create_all(engine)

with engine.connect() as conn:
    conn.execute(pipeline_runs.insert(), [
        {"pipeline": "etl_daily",   "status": "success", "rows_in": 1_500_000, "rows_out": 1_498_200, "duration_s": 42.3},
        {"pipeline": "etl_weekly",  "status": "success", "rows_in": 9_200_000, "rows_out": 9_195_000, "duration_s": 318.7},
        {"pipeline": "etl_monthly", "status": "failed",  "rows_in": 2_100_000, "rows_out": 0,         "duration_s": 8.1},
    ])
    conn.commit()

    rows = conn.execute(
        sa.select(pipeline_runs).order_by(pipeline_runs.c.duration_s.desc())
    ).fetchall()

print("\nPipeline runs:")
for row in rows:
    print(f"  {row.pipeline:<14} {row.status:<8} rows_in={row.rows_in:>10,}  {row.duration_s:.1f}s")
print("✅ SQLAlchemy OK")

## 7. Pydantic

In [ ]:
from pydantic import BaseModel, field_validator, model_validator
from typing import Optional
from datetime import datetime
import pydantic
print(f"pydantic {pydantic.__version__}")

class PipelineConfig(BaseModel):
    name:          str
    version:       str
    source_table:  str
    target_table:  str
    batch_size:    int = 10_000
    max_retries:   int = 3
    active:        bool = True
    tags:          list[str] = []
    created_at:    datetime = datetime.now()

    @field_validator("batch_size")
    @classmethod
    def batch_size_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("batch_size must be positive")
        return v

    @field_validator("version")
    @classmethod
    def version_format(cls, v):
        parts = v.split(".")
        assert len(parts) == 3, "version must be semver (x.y.z)"
        return v

config = PipelineConfig(
    name="etl_customer_events",
    version="2.4.1",
    source_table="raw.customer_events",
    target_table="processed.customer_events",
    batch_size=50_000,
    tags=["production", "critical", "customer-facing"],
)

print("\nPipelineConfig model:")
import json
print(json.dumps(config.model_dump(mode="json"), indent=2, default=str))
print("✅ Pydantic OK")

## 8. Matplotlib

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
print(f"matplotlib {matplotlib.__version__}")

# Sales by region and quarter
quarters = ["Q1", "Q2", "Q3", "Q4"]
regions  = {"North": [120, 145, 190, 230], "South": [95, 88, 110, 140],
             "East":  [200, 220, 195, 260], "West":  [75, 90, 105, 88]}

x     = np.arange(len(quarters))
width = 0.2
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Grouped bar chart
ax = axes[0]
colors_map = ["#185FA5", "#1D9E75", "#BA7517", "#A32D2D"]
for i, (region, values) in enumerate(regions.items()):
    ax.bar(x + i * width, values, width, label=region, color=colors_map[i], alpha=0.85)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(quarters)
ax.set_title("Sales by Region and Quarter", fontsize=12, fontweight="bold")
ax.set_ylabel("Sales ($K)")
ax.legend(framealpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

# Line chart with fill
ax2 = axes[1]
totals = [sum(v[i] for v in regions.values()) for i in range(4)]
ax2.plot(quarters, totals, marker="o", linewidth=2.5, color="#0F6E56", markersize=8)
ax2.fill_between(quarters, totals, alpha=0.15, color="#0F6E56")
ax2.set_title("Total Sales per Quarter", fontsize=12, fontweight="bold")
ax2.set_ylabel("Total Sales ($K)")
ax2.spines[["top", "right"]].set_visible(False)
for i, (q, v) in enumerate(zip(quarters, totals)):
    ax2.annotate(f"{v}", (q, v), textcoords="offset points", xytext=(0, 10),
                 ha="center", fontsize=9, color="#0F6E56")

fig.suptitle("Python 3.15 · Matplotlib 3.10.9 Smoke Test", fontsize=10,
             color="gray", y=1.01)
plt.tight_layout()
plt.show()
print("✅ Matplotlib OK")

## 9. Plotly

In [ ]:
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
print(f"plotly {plotly.__version__}")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Revenue by Quarter", "Package Readiness"),
    specs=[[{"type": "bar"}, {"type": "pie"}]],
)

fig.add_trace(go.Bar(
    x=["Q1", "Q2", "Q3", "Q4"],
    y=[490, 543, 600, 718],
    marker_color=["#185FA5", "#1D9E75", "#BA7517", "#0F6E56"],
    name="Revenue ($K)",
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=["PASS", "INCOMPAT", "SKIP"],
    values=[9, 5, 2],
    marker_colors=["#1D9E75", "#BA7517", "#888780"],
    textinfo="label+percent",
    hole=0.4,
), row=1, col=2)

fig.update_layout(
    title_text="Python 3.15 · Plotly 6.7.0 Smoke Test",
    title_font_size=13,
    showlegend=False,
    height=380,
    margin=dict(t=60, b=20, l=20, r=20),
)
fig.show()
print("✅ Plotly OK")

## 10. Summary

In [ ]:
import importlib

packages = [
    ("numpy",      "np"),
    ("pandas",     "pd"),
    ("polars",     "pl"),
    ("duckdb",     None),
    ("sqlalchemy", "sa"),
    ("pydantic",   None),
    ("matplotlib", None),
    ("plotly",     None),
]

print(f"{'Package':<14} {'Version':<12} {'Status'}")
print("-" * 38)
all_pass = True
for pkg, _ in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "—")
        print(f"  {pkg:<12} {ver:<12} ✅ PASS")
    except ImportError as e:
        print(f"  {pkg:<12} {'—':<12} ❌ FAIL — {e}")
        all_pass = False

print("-" * 38)
print(f"\nPython {sys.version.split()[0]} on {platform.machine()} — ",
      "All checks PASS ✅" if all_pass else "Some checks FAILED ❌")